<a href="https://colab.research.google.com/github/PrasannaMadiwar/LLM-from-Scratch-/blob/main/Entire_GPT_2_140M_Model_fromSrcatch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn
import tiktoken

In [5]:
! pip install tiktoken

In [6]:
class MultiHead(nn.Module):
  def __init__(self,d_in,d_out,context_length,drop_rate,n_heads,qkv_bias=False):

    super().__init__()
    assert(d_out % n_heads == 0)

    self.d_out = d_out
    self.n_heads = n_heads
    self.head_d = d_out // n_heads

    self.query = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.key = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.value = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.dropout = nn.Dropout(drop_rate)

    self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length),diagonal=1))


  def forward(self,x):
    b,n_tokens,d_in = x.shape

    keys = self.key(x)
    values = self.value(x)
    queries = self.query(x)

    keys = keys.view(b,n_tokens,self.n_heads,self.head_d)
    values = values.view(b,n_tokens,self.n_heads,self.head_d)
    queries = queries.view(b,n_tokens,self.n_heads,self.head_d)

    keys = keys.transpose(1,2)
    values = values.transpose(1,2)
    queries = queries.transpose(1,2)

    attn_score = queries @ keys.transpose(2,3)
    mask_bool = self.mask.bool()[:n_tokens,:n_tokens]
    attn_score.masked_fill(mask_bool,-torch.inf)
    attn_weights = torch.softmax(attn_score / keys.shape[-1]**0.5,dim=-1)
    attn_weights = self.dropout(attn_weights)

    context_vec = (attn_weights @ values).transpose(1,2)
    context_vec = context_vec.contiguous().view(b,n_tokens,self.d_out)
    return context_vec





In [7]:
class LayerNorm(nn.Module):

  def __init__(self,d_in):
    super().__init__()
    self.esp = 1e-5
    self.scale = torch.ones(d_in)
    self.shift = torch.zeros(d_in)

  def forward(self,x):
    mean = x.mean(dim=-1,keepdim=True)
    var = x.var(dim=-1,keepdim=True)
    norm_x = (x - mean) / torch.sqrt(var + self.esp)

    return self.scale * norm_x + self.shift


In [8]:
class Gelu(nn.Module):
  def __init__(self):
    super().__init__()
  def forward(self,x):
    return 0.5 * x * (1 + torch.tanh(x * 0.7978845608 * (1 + 0.044715 * x * x)))

In [9]:
class FeedForward(nn.Module):
  def __init__(self,cfg):
    super().__init__()

    self.layers = nn.Sequential(
        nn.Linear(cfg['d_in'],cfg['d_in']*4),
        Gelu(),
        nn.Linear(cfg['d_in']*4,cfg['d_in'])
    )
  def forward(self,x):
    return self.layers(x)

In [10]:
class TransFormer(nn.Module):
  def __init__(self,cfg):
    super().__init__()

    self.norm1 = LayerNorm(cfg['d_in'])
    self.norm2 = LayerNorm(cfg['d_in'])
    self.M_attn = MultiHead(cfg['d_in'],cfg['d_out'],cfg['context_length'],cfg['drop_rate'],cfg['n_heads'])
    self.ff = FeedForward(cfg)
    self.drop_out = nn.Dropout(cfg['drop_rate'])

  def forward(self,x):
    shortcut = x
    x = self.norm1(x)
    x = self.M_attn(x)
    x = self.drop_out(x)
    x = shortcut + x

    shortcut = x
    x = self.norm2(x)
    x = self.ff(x)
    x =  self.drop_out(x)
    x = shortcut + x

    return x



In [11]:
class GPT_140M(nn.Module):
  def __init__(self,cfg):

    super().__init__()

    self.word_emb = nn.Embedding(cfg['vocab_size'],cfg['d_out'])
    self.pos_emb = nn.Embedding(cfg['context_length'],cfg['d_out'])
    self.dropout = nn.Dropout(cfg['drop_rate'])
    self.trf = nn.Sequential(
        *[TransFormer(cfg) for i in range(cfg['n_layers'])]
    )
    self.final_norm = LayerNorm(cfg['d_out'])
    self.output_head = nn.Linear(cfg['d_out'],cfg['vocab_size'],bias=False)

  def forward(self,ind_x):
    b,n_seq =  ind_x.shape
    tok_emb = self.word_emb(ind_x)
    pos_emb = self.pos_emb(torch.arange(n_seq,device=ind_x.device))
    x = tok_emb + pos_emb
    x = self.dropout(x)
    x = self.trf(x)
    x = self.final_norm(x)
    logits = self.output_head(x)
    return logits

In [12]:
cfg = {
    'vocab_size':50257,
    'd_out':768,
    'd_in':768,
    'context_length':1024,
    'drop_rate':0.1,
    'n_heads':12,
    'n_layers':12,
}

In [13]:
model = GPT_140M(cfg)

In [14]:
def get_words(model,idx,max_new,context_length):
  for i in range(max_new):
    idx_cond = idx[:,-context_length:]
    logits = model(idx_cond)
    prob = torch.softmax(logits[:,-1,:],dim=-1)
    id_new = torch.argmax(prob,dim=-1,keepdim=True)
    idx = torch.cat((idx,id_new),dim=1)
  return idx

In [15]:
text = "Hello i am your friend"
tokenizer = tiktoken.get_encoding("gpt2")

In [16]:
encode = tokenizer.encode(text)
encode_tensors = torch.tensor(encode).unsqueeze(0)

In [19]:
model.eval()
out = get_words(model,encode_tensors,400,4)

In [20]:
out = out.squeeze(0).tolist()
tokenizer.decode(out)

'Hello i am your friend Scar GDrg Options workaroundotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend E

Hello i am your friend Scar GDrg Options workaroundotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav genetics Reflex648 Vide Cassandraotericdestruct desolate eligibilityreleasedWestushi inventor enhances recover Wend Economist727 initially brav